In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.linear_model import LinearRegression

# Paths
raw_data_path = "/Users/khoale/Downloads/Alzheimer_Code/adni_mri_ucsf_merged.csv"
output_path = "/Users/khoale/Downloads/Alzheimer_Code/adni_mri_sustain_prepared.csv"

# Load the raw dataset
df = pd.read_csv(raw_data_path)
print(f"Loaded raw data: {df.shape[0]} subjects, {df.shape[1]} columns")

Loaded raw data: 577 subjects, 355 columns


In [2]:
# Map columns belonging to each of our 12 bilateral biomarkers
biomarker_mapping = {
    # --- CORTICAL LOBES ---
    "Frontal": [
        "ST15CV", "ST25CV", "ST36CV", "ST39CV", "ST43CV", "ST45CV", "ST46CV", "ST47CV", "ST51CV", "ST55CV", "ST56CV", # Left
        "ST74CV", "ST84CV", "ST95CV", "ST98CV", "ST102CV", "ST104CV", "ST105CV", "ST106CV", "ST110CV", "ST114CV", "ST115CV" # Right
    ],
    "Temporal": [
        "ST13CV", "ST24CV", "ST26CV", "ST32CV", "ST40CV", "ST44CV", "ST58CV", "ST60CV", "ST62CV", # Left
        "ST72CV", "ST83CV", "ST85CV", "ST91CV", "ST99CV", "ST103CV", "ST117CV", "ST119CV", "ST121CV" # Right
    ],
    "Parietal": [
        "ST31CV", "ST49CV", "ST52CV", "ST57CV", "ST59CV", # Left
        "ST90CV", "ST108CV", "ST111CV", "ST116CV", "ST118CV" # Right
    ],
    "Occipital": [
        "ST23CV", "ST35CV", "ST38CV", "ST48CV", # Left
        "ST82CV", "ST94CV", "ST97CV", "ST107CV" # Right
    ],
    "Cingulate": [
        "ST14CV", "ST34CV", "ST50CV", "ST54CV", # Left
        "ST73CV", "ST93CV", "ST109CV", "ST113CV" # Right
    ],
    "Insula": [
        "ST129CV", "ST130CV"
    ],
    # --- SUBCORTICAL STRUCTURES ---
    "Hippocampus": ["ST29SV", "ST88SV"],
    "Amygdala": ["ST12SV", "ST71SV"],
    "Caudate": ["ST16SV", "ST75SV"],
    "Pallidum": ["ST42SV", "ST101SV"],
    "Putamen": ["ST53SV", "ST112SV"],
    "Accumbens": ["ST11SV", "ST70SV"]
}

In [3]:
# 1. Force all ROI columns and ICV (ST10CV) to numeric values
all_roi_cols = [col for cols in biomarker_mapping.values() for col in cols]
cols_to_convert = all_roi_cols + ["ST10CV"]

for col in cols_to_convert:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 2. Drop rows where Intracranial Volume (ICV) is missing
df = df.dropna(subset=["ST10CV"]).copy()

# 3. Sum left and right hemispheres to get bilateral volumes
aggregated_df = pd.DataFrame()
aggregated_df["PTID"] = df["PTID"]
aggregated_df["Label"] = df["Label"]
aggregated_df["ICV"] = df["ST10CV"]

for region, cols in biomarker_mapping.items():
    # If any sub-region is missing (NaN), pandas sum(axis=1) ignores it. 
    # But if ALL are NaN, we keep it as NaN.
    aggregated_df[region] = df[cols].sum(axis=1, min_count=1)

print(f"Aggregated data shape: {aggregated_df.shape}")
aggregated_df.head()

Aggregated data shape: (577, 15)


,PTID,Label,ICV,Frontal,Temporal,Parietal,Occipital,Cingulate,Insula,Hippocampus,Amygdala,Caudate,Pallidum,Putamen,Accumbens
0,003_S_1057,pMCI,1.352138e+06,147125.0,98226.0,85571.0,39715.0,16328.0,11816.0,6691.5,2213.6,6085.9,3742.3,9163.2,766.7
1,003_S_1059,AD,1.520293e+06,117837.0,79681.0,80119.0,41337.0,14356.0,12163.0,4892.0,1600.6,6365.4,3012.8,5910.4,570.0
2,003_S_1257,AD,1.951182e+06,146946.0,92025.0,91736.0,46878.0,13984.0,13457.0,8906.4,3286.6,10871.2,3684.5,7464.6,730.1
3,003_S_4119,CN,1.475697e+06,166067.0,110048.0,103633.0,43037.0,18874.0,14975.0,7494.9,3273.0,6754.9,3591.5,8843.0,788.4
4,003_S_4152,AD,1.383684e+06,141920.0,85216.0,83363.0,54514.0,18958.0,13464.0,7153.7,2506.5,5036.5,4426.0,7886.8,968.0


In [4]:
# Create a copy to store ICV-normalized volumes
normalized_df = aggregated_df.copy()

# Define the control (CN) cohort to fit the ICV regression line
cn_controls = aggregated_df[aggregated_df["Label"] == "CN"].copy()

print(f"Fitting ICV regression using {len(cn_controls)} CN controls...")

regions = list(biomarker_mapping.keys())

for region in regions:
    # Drop NaNs to fit regression
    clean_cn = cn_controls.dropna(subset=[region, "ICV"])
    
    X_cn = clean_cn[["ICV"]].values
    y_cn = clean_cn[region].values
    
    # Fit linear model: Region = beta * ICV + alpha
    model = LinearRegression()
    model.fit(X_cn, y_cn)
    
    # Calculate residuals for ALL subjects: residual = actual - predicted
    X_all = aggregated_df[["ICV"]].values
    y_all = aggregated_df[region].values
    
    predicted_all = model.predict(X_all)
    residuals = y_all - predicted_all
    
    # Add back the mean volume of controls to keep residuals in a readable volumetric scale
    mean_volume_cn = y_cn.mean()
    normalized_df[region] = residuals + mean_volume_cn

print("ICV correction complete.")

Fitting ICV regression using 186 CN controls...
ICV correction complete.


In [5]:
sustain_ready_df = pd.DataFrame()
sustain_ready_df["PTID"] = normalized_df["PTID"]
sustain_ready_df["Label"] = normalized_df["Label"]

# Fit z-score baseline using the ICV-normalized controls
cn_normalized = normalized_df[normalized_df["Label"] == "CN"]

for region in regions:
    mean_cn = cn_normalized[region].mean()
    std_cn = cn_normalized[region].std() + 1e-9
    
    # Formula for atrophy: Z = (mean_healthy - actual) / std_healthy
    # This ensures shrinking regions produce positive Z-scores
    sustain_ready_df[region] = (mean_cn - normalized_df[region]) / std_cn

sustain_ready_df.head()

,PTID,Label,Frontal,Temporal,Parietal,Occipital,Cingulate,Insula,Hippocampus,Amygdala,Caudate,Pallidum,Putamen,Accumbens
0,003_S_1057,pMCI,-0.028875,-0.261385,1.071885,0.693829,0.380701,0.616037,0.910978,1.549845,0.356920,-0.211728,-0.913232,0.448928
1,003_S_1059,AD,2.948447,2.734552,2.300462,0.714627,2.092270,0.975325,3.623341,3.305170,0.399519,1.683169,2.599112,1.670363
2,003_S_1257,AD,2.333867,2.893539,2.593221,0.479843,3.792096,1.604344,-0.887841,-0.085341,-3.813317,1.331716,2.210264,0.972728
3,003_S_4119,CN,-1.043663,-1.255907,-0.488030,0.265949,-0.653936,-1.212011,0.079457,-0.817398,-0.140538,0.391213,-0.297607,0.388400
4,003_S_4152,AD,0.506762,1.497574,1.434627,-2.317767,-1.019541,-0.456879,0.357072,0.891148,1.621541,-1.537726,0.378157,-0.694718


In [6]:
# 1. Fill missing values (NaNs) with 0 (meaning normal/non-atrophied)
sustain_ready_df[regions] = sustain_ready_df[regions].fillna(0)

# 2. Clamp negative z-scores to 0
# SuStaIn Models only positive deviations from normality.
# A negative Z means the subject's brain region is larger than the control mean (normal).
sustain_ready_df[regions] = sustain_ready_df[regions].clip(lower=0)

print("Preprocessing complete! Preview of final z-scores:")
sustain_ready_df[regions].describe()

Preprocessing complete! Preview of final z-scores:


,Frontal,Temporal,Parietal,Occipital,Cingulate,Insula,Hippocampus,Amygdala,Caudate,Pallidum,Putamen,Accumbens
count,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000
mean,0.857553,1.271560,0.969345,0.628520,0.760675,0.750084,1.209238,1.069409,0.478595,0.504107,0.664636,0.677717
std,1.273613,1.581302,1.320384,0.857758,1.252098,1.174912,1.240608,1.107512,0.677054,0.815674,1.027609,0.736228
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.431077,0.868153,0.438021,0.269144,0.255815,0.375533,0.937780,0.844358,0.177615,0.189345,0.345521,0.442007
75%,1.304427,1.866055,1.540056,1.059351,1.130496,1.056586,2.058219,1.800746,0.761013,0.792398,0.968305,1.213149
max,12.199261,14.310146,11.968191,6.651013,11.232974,10.952114,8.769142,7.423211,6.237394,8.052279,11.033013,3.782354


In [7]:
# Save the prepared file
sustain_ready_df.to_csv(output_path, index=False)
print(f"SuStaIn-ready dataset exported to: {output_path}")

SuStaIn-ready dataset exported to: /Users/khoale/Downloads/Alzheimer_Code/adni_mri_sustain_prepared.csv


In [8]:
# In the model script, you can load the data directly:
import pandas as pd
df = pd.read_csv("adni_mri_sustain_prepared.csv")

# Extract only the 12 biomarker columns for SuStaIn
regions = ["Frontal", "Temporal", "Parietal", "Occipital", "Cingulate", "Insula", 
           "Hippocampus", "Amygdala", "Caudate", "Pallidum", "Putamen", "Accumbens"]
data = df[regions].values